# 03 - Benchmark models

Everything in this project is measured against simple benchmarks, so it matters that the
benchmarks are implemented properly and evaluated under exactly the same rules as the
more complicated models.

## Forecasting design

The assignment specifies a 24-hour forecast horizon and recommends the final 14 days as
the test period. Those two things are not the same thing, and how they are combined
changes the answer substantially.

We treat the 336 test observations as **fourteen consecutive 24-hour-ahead forecasts**.
At each origin the model sees everything observed up to that point and forecasts the next
day. This is a rolling-origin evaluation, and it is the design that matches the stated
task: every reported number describes performance at the 24-hour horizon.

The alternative, forecasting all 336 points from a single origin two weeks out, is run
separately at the end of this notebook so the two can be compared.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, pipeline

hourly = data.load_hourly()
y = hourly[config.TARGET]
train, test = data.train_test_split(y)

print(f"{config.N_ORIGINS} origins x {config.HORIZON}-hour horizon = {config.TEST_STEPS} test points")

In [ ]:
origins = pipeline.rolling_origins(y)

print("Forecast origins (last observation available at each):")
for origin in origins:
    print(" ", origin)

Note that every origin falls at 18:00. This is a direct consequence of issuing one
forecast per day, and it has a consequence for the error analysis that is picked up in
notebook 07: step *h* of the horizon always lands on hour *(18 + h) mod 24*, so error by
horizon and error by hour of day cannot be separated in this design.

## The benchmarks

Five are required by the brief. We add a sixth, an hour-of-week mean profile, because it
is the natural low-variance competitor to the seasonal naive forecast and turns out to be
the strongest of the set.

In [ ]:
from appliance_energy.models import benchmarks

demo = benchmarks.all_benchmarks(train, config.HORIZON, test.index[:config.HORIZON])

pd.DataFrame(demo).assign(actual=test.iloc[:config.HORIZON]).round(1).head(12)

The distinction between the two seasonal naive variants matters here. The daily version
copies yesterday, so it inherits yesterday's idiosyncratic spikes. The weekly version
copies the same day last week. The mean profile averages every past observation in the
same hour-of-week slot, which throws away information about the recent level but is far
less sensitive to any single unusual day.

## Rolling-origin evaluation

In [ ]:
forecasts = pipeline.rolling_benchmarks(y)

results = evaluation.evaluate_all(forecasts, y_true=test, y_train=train)
results.round(3)

The ordering is informative.

`naive` and `drift` are hopeless, as expected: forecasting the next 24 hours by holding
the 18:00 value constant ignores the entire daily cycle. Both have MASE above 1.6 and a
large positive bias, because 18:00 is near the daily peak and they carry that level
through the night.

The `mean` forecast does better than either, at 0.941, purely because it does not commit
to a level.

The three seasonal methods are the serious contenders, and they rank in the opposite
order to what naive intuition suggests: the daily seasonal naive (0.904) is worse than the
weekly (0.813), which is worse than the hour-of-week mean profile (0.712). Averaging beats
copying, and copying last week beats copying yesterday.

That ranking says something concrete about the household: day-to-day behaviour is noisy
enough that yesterday is a poor guide to today, but there is a stable average weekly
rhythm underneath the noise.

## Why the mean profile wins

In [ ]:
profile = forecasts["seasonal_mean_profile"]
weekly = forecasts["seasonal_naive_weekly"]

print(f"Standard deviation of the actual test series:      {test.std():.1f}")
print(f"Standard deviation of the weekly naive forecast:   {weekly.std():.1f}")
print(f"Standard deviation of the mean profile forecast:   {profile.std():.1f}")

The weekly seasonal naive forecast is about as variable as the series itself, because it
is a copy of a real week. The mean profile is much smoother. On a series where the timing
of peaks is close to unpredictable, being smooth is an advantage: a forecast that puts a
peak in the wrong hour is penalised twice, once for the peak that did not happen and once
for the peak that did.

This is the central tension of the whole project, and it is why the more sophisticated
models struggle to pull away from a simple average.

## Comparison with a single 336-step forecast

In [ ]:
single_block = benchmarks.all_benchmarks(train, config.TEST_STEPS, test.index)
single_results = evaluation.evaluate_all(single_block, y_true=test, y_train=train)

comparison = (
    results.set_index("model")[["MASE"]]
    .rename(columns={"MASE": "rolling_origin"})
    .join(single_results.set_index("model")[["MASE"]].rename(columns={"MASE": "single_block"}))
)

comparison.round(3)

The seasonal methods are almost unaffected by the change of design, because they repeat a
fixed pattern regardless of how far ahead they are asked to forecast. `naive` and `drift`
collapse completely, from about 1.6 to nearly 5.

This is worth stating explicitly in the report: a large part of the apparent difference
between "good" and "bad" benchmarks in a single-block evaluation is an artefact of the
evaluation design rather than a property of the methods. Under the rolling-origin design
the naive forecast is refreshed every day and is merely bad; under the single-block design
it is asked to extrapolate a single 18:00 reading across a fortnight and is catastrophic.

**The strongest benchmark is the hour-of-week mean profile, at MASE 0.712.** That is the
number every later model has to beat.